# CorridorIQ — Baseline Training Walkthrough

This notebook is a step-by-step explanation of `code/scripts/train_baseline.py`.  
You can run each cell individually to understand what it does before moving to the next.

### What does this script try to prove?

> **"Can a model learn from historical payment routing decisions — and beat the human ops team?"**

The system trains three things:
1. A **route ranker** — given a new payment and 4–6 candidate routes, which route should be top-ranked?  
2. A **time-to-credit corrector** — adjust the simulator's estimated hours with actual history.  
3. A **cost predictor** — predict the all-in cost (fees + FX margin) the payment will actually incur.

All models use [LightGBM](https://lightgbm.readthedocs.io/), a gradient-boosted tree library.

---
## Step 1 — Imports & Path Setup

In [1]:
import sqlite3
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

# Resolve the DB path relative to this notebook's location
# notebooks/ is two levels below the project root, same as scripts/
DATA_DIR = Path("__file__").resolve().parents[2] / "Data" / "corridoriq_history"

# Override with a direct relative path that works when running from the notebook itself
DB_PATH = Path("../../Data/corridoriq_history/corridoriq_history.db").resolve()
print("DB path:", DB_PATH)
print("Exists:", DB_PATH.exists())

DB path: /Users/apple/projects/PaymentsCorridorPrediction/Data/corridoriq_history/corridoriq_history.db
Exists: True


---
## Step 2 — Load the Training View

The database contains a pre-built view called **`v_ranking_training`**.  
It is a join of the `route_candidate` and `payment` tables with all the columns we need.  
Each row represents **one candidate route for one payment**.

We only load rows where `eligible = 1` — routes that were actually available to ops at the time  
(some routes are excluded for a payment because of per-transfer caps or client-type restrictions).

In [28]:
conn = sqlite3.connect(DB_PATH)

df_temp = pd.read_sql("SELECT * FROM v_ranking_training WHERE eligible = 1 and payment_id in (1, 2, 3, 46, 16) order by payment_id, relevance", conn)
print(f"Rows loaded      : {len(df_temp):,}")
print(f"Unique payments  : {df_temp.payment_id.nunique():,}")
print(f"Unique routes/row: {df_temp.template_id.nunique()} templates")
print(f"Date range       : {df_temp.value_date.min()} → {df_temp.value_date.max()}")
df_temp.head(30)

Rows loaded      : 25
Unique payments  : 5
Unique routes/row: 6 templates
Date range       : 2024-04-01 → 2024-04-01


,payment_id,corridor_id,value_date,template_id,bene_bank_id,client_type,amount_usd,charges,purpose_code,priority,...,calendar_conflict,cutoff_slack_hours,max_risk,stp_rate_nominal,realized_hours,realized_cost_usd,realized_outcome,utility,relevance,was_chosen
0,1,USD-INR,2024-04-01,EXP,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.972,17.81,62.20,Credited,0.9188,0,0
1,1,USD-INR,2024-04-01,LDN,IN_B2,Retail,580.0,OUR,OTHR,15,...,1,6.98,2,0.955,43.29,59.57,Credited,0.9338,0,0
2,1,USD-INR,2024-04-01,ASIA,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.948,19.77,51.28,Credited,0.8593,1,0
3,1,USD-INR,2024-04-01,GULF,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.931,44.40,46.93,Credited,0.8569,2,0
4,1,USD-INR,2024-04-01,PAY,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.986,2.18,7.90,Credited,0.1776,3,0
5,1,USD-INR,2024-04-01,ECO,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.962,212.68,4.58,Credited,0.1500,4,1
6,2,USD-INR,2024-04-01,GULF,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.931,46.00,49.28,Credited,0.7039,0,0
7,2,USD-INR,2024-04-01,LDN,IN_B1,Retail,2050.0,SHA,OTHR,70,...,1,6.85,2,0.955,44.84,63.53,Credited,0.7312,0,0
8,2,USD-INR,2024-04-01,ECO,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.962,212.37,6.05,Credited,0.7000,1,0
9,2,USD-INR,2024-04-01,ASIA,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.948,41.36,54.51,Credited,0.6986,2,0


---
i understand that before ML model, in the training data, est cost and time is not calculated by ML model. 
But then how the simulator calculated that? 

Can you please give an example of one record?
--
Let me use two routes for payment_id=1 side by side — PAY (fast) and ECO (slow) — to show exactly how the simulator calculates both.

---

## EST_COST_USD — the formula is exact, not statistical

```
est_cost_usd = fee_total_usd + (amount_usd × fx_bps / 10,000)
```

**PAY route:**
```
= $5.00  +  (580 × 50 / 10,000)
= $5.00  +  $2.90
= $7.90  ✓  (matches the record)
```

**ECO route:**
```
= $4.00  +  (580 × 10 / 10,000)
= $4.00  +  $0.58
= $4.58  ✓  (matches the record)
```

The fees come from **pre-agreed bank contracts**. The FX spread is a **fixed rate per corridor**. Pure lookup, zero statistics.

---

## EST_HOURS — rules-based calendar engine

Start with `naive_hours` (raw processing time, no calendar), then add penalties:

| | PAY route | ECO route |
|---|---|---|
| `naive_hours` | 1.5 h | 87.0 h |
| `est_nonbusiness_days` | 0 | 2 (crosses a weekend) |
| `calendar_conflict` | 0 | 0 |
| `hop_count` | 3 | 5 |
| **`est_hours`** | **1.5 h** | **210.48 h** |

**PAY:** 3 hops, no weekend crossed, no conflicts → simulator adds nothing → stays at 1.5 h

**ECO:** 5 hops, crosses 2 non-business days → each non-business day means the payment sits idle waiting for banks to reopen → 87h balloons to 210h

The simulator essentially asks: *"of the raw processing hours, how many fall on bank-closed days?"* and adds that waiting time. Like a parcel delivery app that knows the warehouse is closed on weekends.

---

**Bottom line:** no statistics, no ML. Just:
- **Cost** = fee schedule + FX rate (from contracts)
- **Time** = raw processing hours + calendar waiting time (from bank timetables)

In [16]:

df = pd.read_sql("SELECT * FROM v_ranking_training WHERE eligible = 1", conn)

print(f"Rows loaded      : {len(df):,}")
print(f"Unique payments  : {df.payment_id.nunique():,}")
print(f"Unique routes/row: {df.template_id.nunique()} templates")
print(f"Date range       : {df.value_date.min()} → {df.value_date.max()}")
df.head(10)

Rows loaded      : 169,240
Unique payments  : 40,000
Unique routes/row: 6 templates
Date range       : 2024-04-01 → 2026-09-30


,payment_id,corridor_id,value_date,template_id,bene_bank_id,client_type,amount_usd,charges,purpose_code,priority,...,calendar_conflict,cutoff_slack_hours,max_risk,stp_rate_nominal,realized_hours,realized_cost_usd,realized_outcome,utility,relevance,was_chosen
0,1,USD-INR,2024-04-01,ASIA,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.948,19.77,51.28,Credited,0.8593,1,0
1,1,USD-INR,2024-04-01,ECO,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.962,212.68,4.58,Credited,0.1500,4,1
2,1,USD-INR,2024-04-01,EXP,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.972,17.81,62.20,Credited,0.9188,0,0
3,1,USD-INR,2024-04-01,GULF,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.931,44.40,46.93,Credited,0.8569,2,0
4,1,USD-INR,2024-04-01,LDN,IN_B2,Retail,580.0,OUR,OTHR,15,...,1,6.98,2,0.955,43.29,59.57,Credited,0.9338,0,0
5,1,USD-INR,2024-04-01,PAY,IN_B2,Retail,580.0,OUR,OTHR,15,...,0,6.98,2,0.986,2.18,7.90,Credited,0.1776,3,0
6,2,USD-INR,2024-04-01,ASIA,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.948,41.36,54.51,Credited,0.6986,2,0
7,2,USD-INR,2024-04-01,ECO,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.962,212.37,6.05,Credited,0.7000,1,0
8,2,USD-INR,2024-04-01,EXP,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.972,18.93,67.79,Credited,0.5946,3,0
9,2,USD-INR,2024-04-01,GULF,IN_B1,Retail,2050.0,SHA,OTHR,70,...,0,6.85,2,0.931,46.00,49.28,Credited,0.7039,0,0


### What are the key columns?

| Column | Meaning |
|--------|---------|
| `payment_id` | Identifies the payment — groups all candidate routes together |
| `template_id` | The route template (EXP, ASIA, LDN, GULF, PAY, ECO) |
| `relevance` | **Label for ranking (0–4).** 4 = the route that would have performed best in reality |
| `was_chosen` | 1 = ops actually picked this route; 0 = alternatives that existed but weren't used |
| `est_hours` | Simulator's calendar-aware estimate of time to credit |
| `naive_hours` | Raw processing hours with no calendar awareness (the "before CorridorIQ" view) |
| `est_cost_usd` | Quoted fees + FX margin |
| `realized_hours` | What actually happened (or would have, for the unchosen routes) |
| `realized_outcome` | Credited / Repaired / Returned |
| `priority` | 0 = cheapest route preferred, 100 = fastest route preferred |

In [18]:
# See the relevance label distribution
print("Relevance label distribution (0=worst, 4=best):")
print(df.relevance.value_counts().sort_index())

print("\nHow many routes per payment (most payments have 4-6 candidates):")
routes_per_payment = df.groupby("payment_id").size()
print(routes_per_payment.value_counts().sort_index())

Relevance label distribution (0=worst, 4=best):
relevance
0    29439
1    24551
2    35250
3    40000
4    40000
Name: count, dtype: int64

How many routes per payment (most payments have 4-6 candidates):
2     4750
3    10699
4     8264
5     3135
6    13152
Name: count, dtype: int64


## Learning — 

in the relevance distribution - 
we have 
0 - 29439
1 - 24551
etc. 

Does that means that in the data, for the relevance of 0 the was_chosen were 29439?

No, that's a different thing. The relevance distribution counts how many route-candidate rows have each relevance label across all 169,240 rows — it has nothing to do with was_chosen.


Looking at the sample data in cell `77f54468`, the two columns are independent:

| row | relevance | was_chosen | meaning |
|-----|-----------|------------|---------|
| ECO (payment 1) | 4 | 1 | Best route *and* ops chose it — lucky alignment |
| LDN (payment 16) | 0 | 1 | Worst route, but ops *still* chose it |
| PAY (payment 2) | 4 | 1 | Best route, ops chose it |

So:
- **`relevance = 0` with count 29,439** means: across the whole dataset, 29,439 candidate rows were the worst-performing route in hindsight (the route that would have had the lowest utility/worst outcome)
- **`was_chosen`** is entirely separate — it flags which route ops actually picked (exactly one `was_chosen=1` per payment)

The fact that relevance 0 and 1 have *fewer* rows than 3 and 4 simply means the data is uneven — some payments had fewer candidates (2–3 routes instead of 6), so there's no "middle" relevance label for them, leaving fewer rows at the bottom tiers.

---
## Step 3 — Feature Engineering

The raw view doesn't have every feature we need in the right form.  
We derive three new columns and encode three ID columns as **categorical** so LightGBM can handle them natively.

In [20]:
# log_amount: payment amount is highly skewed (e.g. $1k retail vs $1M FI).
# Taking log compresses the range and makes it a better feature.
df["log_amount"] = np.log(df.amount_usd)

# is_fi: FI (financial institution) clients behave differently from Retail.
# Encoding as 0/1 is cleaner than passing the string.
df["is_fi"] = (df.client_type == "FI").astype(int)

# tpl / corr / bene: LightGBM can handle categories natively (no one-hot needed).
# It finds the best binary split on the category label integer internally.
df["tpl"]  = df.template_id.astype("category")    # route template
df["corr"] = df.corridor_id.astype("category")    # corridor (e.g. USD-JPY)
df["bene"] = df.bene_bank_id.astype("category")   # beneficiary bank

print("Feature engineering done. Sample new columns:")
df[["amount_usd", "log_amount", "client_type", "is_fi", "tpl", "corr", "bene"]].head(4)

Feature engineering done. Sample new columns:


,amount_usd,log_amount,client_type,is_fi,tpl,corr,bene
0,580.0,6.363028,Retail,0,ASIA,USD-INR,IN_B2
1,580.0,6.363028,Retail,0,ECO,USD-INR,IN_B2
2,580.0,6.363028,Retail,0,EXP,USD-INR,IN_B2
3,580.0,6.363028,Retail,0,GULF,USD-INR,IN_B2


### The full feature list

These are the 22 columns fed into all three models.

In [21]:
FEATURES = [
    # --- route timing estimates ---
    "est_hours",            # simulator's calendar-aware estimate
    "naive_hours",          # raw processing hours, no calendar (what old system showed)
    # --- cost features ---
    "est_cost_usd",         # quoted total cost: fees + FX margin
    "fee_total_usd",        # correspondent fees component only
    "fx_bps",               # FX spread in basis points
    # --- route structure ---
    "hop_count",            # number of banks in the chain
    "intermediaries",       # number of intermediary (non-ordering) banks
    # --- calendar / timing risk ---
    "est_nonbusiness_days", # non-business days the route crosses
    "calendar_conflict",    # 1 = an intermediary is closed while US is open
    "cutoff_slack_hours",   # hours until the ordering bank's cut-off (negative = already missed)
    # --- risk & quality ---
    "max_risk",             # highest jurisdiction risk score on the route (1-3)
    "stp_rate_nominal",     # expected straight-through rate (0-1)
    # --- payment context ---
    "priority",             # 0 = cheapest, 100 = fastest
    "log_amount",           # log of payment USD amount
    "is_fi",                # 1 = financial institution client
    "release_hour_et",      # hour of day the payment was released (ET)
    "release_weekday",      # day of week (0=Sun...6=Sat)
    "day_of_month",         # 1-31 (month-end payments behave differently)
    "has_deadline",         # 1 = client specified an arrive-by date
    # --- entity IDs (categorical) ---
    "tpl",                  # route template
    "corr",                 # corridor
    "bene",                 # beneficiary bank
]

print(f"{len(FEATURES)} features total")

22 features total


---
## Step 4 — Train / Test Split (Time-Based)

We **never** randomly shuffle and split for time-series-style problems.  
That would leak the future into training. Instead we split on a calendar date:

- **Train**: all payments before `2026-04-01` (approx. 2 years of history)
- **Test**: all payments from `2026-04-01` onward (the last ~6 months)

We also sort by `(payment_id, template_id)` so that all candidate routes for a payment  
are contiguous — this is required by LightGBM's LambdaRank which processes groups.

In [22]:
SPLIT_DATE = "2026-04-01"

# Sort so all rows for the same payment are adjacent (required for group-based ranking)
df = df.sort_values(["payment_id", "template_id"])

tr = df[df.value_date < SPLIT_DATE]
te = df[df.value_date >= SPLIT_DATE]

print(f"Train : {tr.payment_id.nunique():,} payments ({len(tr):,} rows)")
print(f"Test  : {te.payment_id.nunique():,} payments ({len(te):,} rows)")
print(f"\nTrain date range : {tr.value_date.min()} → {tr.value_date.max()}")
print(f"Test  date range : {te.value_date.min()} → {te.value_date.max()}")

Train : 30,335 payments (128,332 rows)
Test  : 9,665 payments (40,908 rows)

Train date range : 2024-04-01 → 2026-03-31
Test  date range : 2026-04-01 → 2026-09-30


---
## Step 5 — Evaluation Metrics (defined before training)

We define the two metrics used to judge the ranker **before** training, so we understand what we're optimising for.

### NDCG@K (Normalised Discounted Cumulative Gain at K)

The standard information-retrieval metric for ranked lists.  
- It rewards placing the **most relevant** routes at the top.  
- The `@3` means: we only care about the top 3 positions.  
- Score ranges from 0 (worst) to 1.0 (perfect ranking).  
- **Discounting** means position 1 is worth more than position 2, etc.

### Top-1 Accuracy

The fraction of payments where the model's #1 ranked route  
is *also* the route with the highest relevance label (i.e. the truly best route).

---

Learning

Cell 23 is the **scorecard** — it answers: *"How good is each of the three rankers at picking the best route?"*

---

**What it does in plain English:**

For each of the 9,665 test payments, it looks at all the candidate routes and asks: *"Did this ranker put the best routes at the top?"*

It tests three rankers:

| Ranker | What it is |
|--------|------------|
| **Ops choice** | What the human ops team actually picked historically |
| **Simulator heuristic** | A simple formula (time + cost weighted by priority) — no ML |
| **LightGBM ranker** | The trained ML model |

---

**The two scores:**

- **NDCG@3** — "Did the top 3 routes end up in roughly the right order?" (1.0 = perfect, 0.0 = completely wrong)
- **Picks best** — "What % of the time did the #1 ranked route happen to be the *actually best* route?"

---

**The result:**

```
Ops choice (history)    0.883    80.1%   ← humans were right 80% of the time
Simulator heuristic     0.975    91.6%   ← simple formula beats humans
LightGBM ranker         0.979    94.5%   ← ML beats the formula
```

The key takeaway: **the ML model picks the best route 94.5% of the time**, beating both the human ops team (80.1%) and the pure formula (91.6%). This is the proof that the model is worth deploying.

In [23]:
def ndcg_at_k(groups, k=3):
    """groups: list of (relevance_array, score_array) — one tuple per payment."""
    out = []
    for rel, score in groups:
        rel, score = np.asarray(rel, dtype=float), np.asarray(score, dtype=float)
        # rank routes by score descending, take top-k
        order = np.argsort(-score)[:k]
        # DCG: sum of (2^relevance - 1) / log2(rank + 2)  — rank is 0-indexed here
        dcg  = sum((2 ** rel[i] - 1) / np.log2(r + 2) for r, i in enumerate(order))
        # ideal DCG: sort relevance descending (the best possible ranking)
        ideal = sorted(rel, reverse=True)[:k]
        idcg = sum((2 ** g - 1) / np.log2(r + 2) for r, g in enumerate(ideal))
        out.append(dcg / idcg if idcg > 0 else 1.0)
    return float(np.mean(out))


def top1(groups):
    """Fraction of payments where the top-scored route has the highest relevance."""
    results = []
    for rel, score in groups:
        rel, score = np.asarray(rel, dtype=float), np.asarray(score, dtype=float)
        results.append(rel[np.argmax(score)] == rel.max())
    return float(np.mean(results))


# Quick sanity check: perfect ranker should score 1.0 on both metrics
perfect = [([0, 1, 2, 3], [0, 1, 2, 3])]
print(f"Perfect ranker — NDCG@3: {ndcg_at_k(perfect):.3f}, Top-1: {top1(perfect):.3f}")

# Worst ranker (reversed order)
worst = [([0, 1, 2, 3], [3, 2, 1, 0])]
print(f"Worst  ranker — NDCG@3: {ndcg_at_k(worst):.3f}, Top-1: {top1(worst):.3f}")

Perfect ranker — NDCG@3: 1.000, Top-1: 1.000
Worst  ranker — NDCG@3: 0.227, Top-1: 0.000


---
## Step 6 — Train the LightGBM LambdaRank Ranker

### What is LambdaRank?

LambdaRank is a **learning-to-rank** algorithm.  
Unlike a classifier or regressor, it doesn't predict a class or a number.  
Instead, it learns a **scoring function** that, when applied to all candidates in a group  
(here: all eligible routes for a payment), produces scores whose *ordering* maximises NDCG.

### The `group` parameter

LightGBM needs to know which rows belong to the same payment (so it can rank within groups).  
We pass a list of group sizes — e.g. `[5, 4, 6, 5, ...]` meaning payment 1 has 5 routes, payment 2 has 4, etc.

In [27]:
# Group sizes: how many candidate rows per payment in the training set
train_groups = tr.groupby("payment_id").size().values

print(train_groups[:10])

print(f"Training group sizes (routes per payment): min={train_groups.min()}, max={train_groups.max()}, mean={train_groups.mean():.1f}")

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=400,        # 400 boosting rounds
    learning_rate=0.05,      # small LR = more rounds needed but more stable
    num_leaves=31,           # tree complexity (31 is LightGBM's default)
    min_child_samples=40,    # a leaf needs at least 40 samples — prevents overfitting
    verbose=-1               # suppress training output
)

ranker.fit(
    tr[FEATURES],
    tr.relevance,             # the 0-4 label
    group=train_groups        # group structure
)

print("Ranker trained.")

[6 6 2 5 6 6 3 3 6 6]
Training group sizes (routes per payment): min=2, max=6, mean=4.2
Ranker trained.


### Score the test set

In [29]:
# ranker.predict returns a raw score per row — higher = model thinks this route is better
te = te.assign(model=ranker.predict(te[FEATURES]))

print("Sample model scores vs actual relevance for one payment:")
sample_pid = te.payment_id.iloc[0]
te[te.payment_id == sample_pid][["template_id", "relevance", "was_chosen", "model", "est_hours", "est_cost_usd"]]

Sample model scores vs actual relevance for one payment:


,template_id,relevance,was_chosen,model,est_hours,est_cost_usd
128332,ASIA,0,0,-0.132861,20.92,50.77
128333,ECO,2,0,-0.163166,138.42,4.35
128334,EXP,3,0,1.296794,17.92,61.33
128335,GULF,1,0,-2.049793,68.42,46.56
128336,LDN,0,0,-3.818914,67.42,58.95
128337,PAY,4,1,6.689150,1.50,6.75


---
## Step 7 — The Transparent Heuristic Baseline

Before we can say the model is good, we need something to compare it against.  
The heuristic represents **"MVP 1"** — what CorridorIQ shows before ML is added:
just a formula using the simulator's own estimates.

**Formula:**
```
score = -(w * normalised_log_time + (1 - w) * normalised_log_cost)
```
- `w = priority / 100`  →  a priority-100 payment weights time 100%, cost 0%
- Both time and cost are **normalised within each payment** (so the cheapest route is 0.0, most expensive is 1.0)
- Negated so higher score = lower (time+cost) = better route
- Uses only `est_hours` and `est_cost_usd` — the simulator's outputs, no ML

In [30]:
def heuristic_score(group_df):
    lt = np.log(group_df.est_hours.clip(lower=0.5))  # log time; clip at 0.5 h to avoid log(0)
    lc = np.log(group_df.est_cost_usd)               # log cost

    # Normalise to [0, 1] within this payment's candidates
    tn = (lt - lt.min()) / (lt.max() - lt.min()) if lt.max() > lt.min() else lt * 0
    cn = (lc - lc.min()) / (lc.max() - lc.min()) if lc.max() > lc.min() else lc * 0

    w = group_df.priority.iloc[0] / 100   # weight: 0=cheapest, 1=fastest
    return -(w * tn + (1 - w) * cn)       # negate: higher = better


te = te.assign(heur=te.groupby("payment_id", group_keys=False).apply(heuristic_score))

print("Heuristic score added. Example:")
te[te.payment_id == sample_pid][["template_id", "relevance", "was_chosen", "heur", "model"]]

Heuristic score added. Example:


,template_id,relevance,was_chosen,heur,model
128332,ASIA,0,0,-0.686253,-0.132861
128333,ECO,2,0,-0.700000,-0.163166
128334,EXP,3,0,-0.683731,1.296794
128335,GULF,1,0,-0.859755,-2.049793
128336,LDN,0,0,-0.884228,-3.818914
128337,PAY,4,1,-0.049813,6.689150


---
## Step 8 — Compare All Three Rankers

We compare:
1. **Ops choice (history)** — use `was_chosen` as the score. Measures how good the human decisions were.
2. **Simulator heuristic** — the formula from Step 7.
3. **LightGBM ranker** — the trained model.

The `gs()` helper builds the `(relevance, score)` tuples for every test payment.

In [15]:
def make_groups(score_col):
    """Returns list of (relevance_array, score_array) for each payment with >1 candidate."""
    return [
        (g.relevance.values, g[score_col].values.astype(float))
        for _, g in te.groupby("payment_id")
        if len(g) > 1
    ]

print(f"Test payments : {te.payment_id.nunique():,}  (from {SPLIT_DATE})")
print()
print(f"{'':30}{'NDCG@3':>8}{'Picks best':>12}")
print("-" * 52)
for name, col in [
    ("Ops choice (history)", "was_chosen"),
    ("Simulator heuristic",  "heur"),
    ("LightGBM ranker",      "model"),
]:
    g = make_groups(col)
    ndcg = ndcg_at_k(g)
    t1   = top1(g) * 100
    print(f"{name:30}{ndcg:8.3f}{t1:11.1f}%")

Test payments : 9,665  (from 2026-04-01)

                                NDCG@3  Picks best
----------------------------------------------------
Ops choice (history)             0.883       80.1%
Simulator heuristic              0.975       91.6%
LightGBM ranker                  0.979       94.5%


### How to read these results

| Score | Interpretation |
|-------|----------------|
| NDCG@3 close to **1.0** | Near-perfect ranking in the top 3 positions |
| NDCG@3 close to **0.9** | Strong — top 3 is mostly in the right order |
| NDCG@3 below **0.8** | Weak — a lot of worse routes are ranked above better ones |

**Expected outcome**: LightGBM should beat ops choice and heuristic on NDCG@3,  
because it learns from corridor-specific and time-of-day patterns that the heuristic ignores.

---
## Step 9 — Feature Importance (bonus explainability)

Which features does the ranker actually use most?

In [16]:
importance = pd.Series(
    ranker.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("Top 10 features by split gain:")
print(importance.head(10).to_string())

Top 10 features by split gain:
priority              1947
est_hours             1864
est_cost_usd          1598
log_amount            1152
day_of_month           846
cutoff_slack_hours     592
naive_hours            547
release_hour_et        527
release_weekday        469
fee_total_usd          419


---
## Step 10 — Time-to-Credit Regressor

### Goal
Predict how many hours a payment will actually take — **better than the simulator's own estimate**.

### The log-ratio trick

Instead of predicting `realized_hours` from scratch, the model predicts:
```
log(realized_hours / est_hours)
```
This is the **correction factor** relative to the simulator.  
- If the model predicts 0, the simulator was perfect.  
- If it predicts +0.5, the real time was `e^0.5 ≈ 1.65×` longer.  
- If it predicts -0.3, the real time was `e^{-0.3} ≈ 0.74×` shorter.

**Why?** The simulator is already a pretty good estimate. Learning the *residual* is much easier  
than learning the absolute hours from scratch. It also keeps the prediction in a log-normal space  
which handles the heavy tail of very long payments gracefully.

### Filter: returned payments excluded

Payments that were returned never credited — their `realized_hours` includes a 72-hour penalty  
and represents the return time, not the credit time. We exclude them.

In [17]:
ok = df.realized_outcome != "Returned"

tr_r = tr[ok.loc[tr.index]]   # training rows where the payment credited or was repaired
te_r = te[ok.loc[te.index]]   # test rows (same filter)

print(f"Training rows after filtering returned: {len(tr_r):,} (excluded {len(tr) - len(tr_r):,})")

# The log-ratio target
ratio = np.log(
    tr_r.realized_hours.clip(lower=0.25) /    # clip to avoid log(0)
    tr_r.est_hours.clip(lower=0.25)
)

print(f"\nLog-ratio (target) stats:")
print(f"  mean  = {ratio.mean():.3f}  (0 = simulator is unbiased on average)")
print(f"  std   = {ratio.std():.3f}")
print(f"  min   = {ratio.min():.3f}")
print(f"  max   = {ratio.max():.3f}")

Training rows after filtering returned: 127,404 (excluded 928)

Log-ratio (target) stats:
  mean  = 0.104  (0 = simulator is unbiased on average)
  std   = 0.300
  min   = -2.342
  max   = 3.518


In [18]:
# MAE (L1) objective is more robust to outlier payments than MSE
reg_time = lgb.LGBMRegressor(
    objective="l1",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,       # deeper trees than the ranker — regression can handle more complexity
    verbose=-1
).fit(tr_r[FEATURES], ratio)

print("Time-to-credit regressor trained.")

Time-to-credit regressor trained.


In [19]:
# To get predicted hours: apply the correction to the simulator estimate
pred_hours = te_r.est_hours.clip(lower=0.25) * np.exp(reg_time.predict(te_r[FEATURES]))

# Only evaluate on routes under 72 h (excludes extreme edge cases and batched routes)
under72 = te_r.realized_hours < 72

mae_simulator = np.mean(np.abs(te_r.est_hours - te_r.realized_hours)[under72])
mae_model     = np.mean(np.abs(pred_hours       - te_r.realized_hours)[under72])

print(f"Time to credit MAE (routes < 72 h)")
print(f"  Simulator baseline : {mae_simulator:.1f} h")
print(f"  LightGBM model     : {mae_model:.1f} h")
print(f"  Improvement        : {(mae_simulator - mae_model) / mae_simulator * 100:.1f}%")

Time to credit MAE (routes < 72 h)
  Simulator baseline : 3.6 h
  LightGBM model     : 3.4 h
  Improvement        : 5.2%


---
## Step 11 — All-in Cost Regressor

Predict the actual all-in cost (fees + FX spread) in USD.  
Here we predict the absolute cost directly, not a ratio, since `est_cost_usd`  
already has a different structure than hours (no log-ratio trick needed).

Metric: **MAPE** (Mean Absolute Percentage Error) — easier to interpret across payment sizes.

In [20]:
reg_cost = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1
).fit(tr_r[FEATURES], tr_r.realized_cost_usd)

pred_cost = reg_cost.predict(te_r[FEATURES])

mape_cost = np.mean(
    np.abs(pred_cost - te_r.realized_cost_usd) / te_r.realized_cost_usd
) * 100

print(f"All-in cost MAPE: {mape_cost:.1f}%")
print("(~5-10% is considered good for this type of payment cost prediction)")

All-in cost MAPE: 4.6%
(~5-10% is considered good for this type of payment cost prediction)


---
## Step 12 — Full Results Summary

This reproduces the final printed output of the original script.

In [21]:
print(f"Train payments {tr.payment_id.nunique():,} · test payments {te.payment_id.nunique():,} (from {SPLIT_DATE})")
print()
print(f"{'':30}{'NDCG@3':>8}{'Picks best':>12}")
print("-" * 52)
for name, col in [
    ("Ops choice (history)", "was_chosen"),
    ("Simulator heuristic",  "heur"),
    ("LightGBM ranker",      "model"),
]:
    g = make_groups(col)
    print(f"{name:30}{ndcg_at_k(g):8.3f}{top1(g) * 100:11.1f}%")

print()
print(f"Time to credit MAE (routes < 72 h): simulator {mae_simulator:.1f} h · LightGBM {mae_model:.1f} h")
print(f"All-in cost MAPE: LightGBM {mape_cost:.1f}%")

Train payments 30,335 · test payments 9,665 (from 2026-04-01)

                                NDCG@3  Picks best
----------------------------------------------------
Ops choice (history)             0.883       80.1%
Simulator heuristic              0.975       91.6%
LightGBM ranker                  0.979       94.5%

Time to credit MAE (routes < 72 h): simulator 3.6 h · LightGBM 3.4 h
All-in cost MAPE: LightGBM 4.6%


---
## Summary — What did we just do?

| Step | What happened |
|------|---------------|
| Load data | Pulled 169K rows from `v_ranking_training`, each row = one route option for one payment |
| Feature engineering | Added `log_amount`, `is_fi`, and three category-encoded IDs |
| Train/test split | Time-based split at 2026-04-01 — no data leakage |
| Ranker | LightGBM LambdaRank learns to rank routes by relevance label within each payment |
| Heuristic | Priority-weighted log(time + cost) formula — the pre-ML baseline |
| Evaluation | NDCG@3 and Top-1 accuracy — compared model, heuristic, and historical ops choices |
| Time regressor | Predicts log(real/estimated) correction factor; applied as a multiplier to the simulator |
| Cost regressor | Predicts absolute all-in cost; evaluated with MAPE |

**Next step:** if the LightGBM ranker beats ops choice and the heuristic, it's ready to be wired into the  
CorridorIQ engine (`engine.py`) as the live route-recommendation model.

---
Few Questions 

Let me use the PAY route from payment_id=1 as the example record.

---

**The record going into the tree:**

```
priority       = 15
est_hours      = 1.5
est_cost_usd   = 7.90
log_amount     = 6.36  (log of $580)
release_hour   = 9.03
template       = PAY
...
```

---

**How one decision tree works — like a flowchart:**

```
                    priority > 50?
                   /              \
                 NO                YES
                 |                  |
          est_hours < 10?       est_hours < 5?
          /         \            /          \
        NO           YES       NO            YES
        |             |         |              |
     score=-2      score=+3  score=+1       score=+5
```

Our record has `priority=15` → goes LEFT.
Then `est_hours=1.5 < 10` → goes RIGHT.
**Lands at score = +3** (this route looks good).

---

**LightGBM doesn't use one tree — it uses 400.**

Each tree corrects the mistakes of the previous one:

```
Tree 1 says: +3.0
Tree 2 notices tree 1 was slightly off, adds: +0.2
Tree 3 notices tree 2 was still off, adds: +0.1
...
Tree 400: tiny final nudge
─────────────────────────
Final score for PAY route: +6.69
```

That +6.69 is exactly what you saw in the model output — the highest score, correctly identifying PAY as the best route.

---

**One sentence summary:**

> Each tree splits routes into groups by asking yes/no questions about the features, 400 trees vote, and the total score tells you how good that route is.

Let me use two routes for payment_id=1 side by side — PAY (fast) and ECO (slow) — to show exactly how the simulator calculates both.

---

## EST_COST_USD — the formula is exact, not statistical

```
est_cost_usd = fee_total_usd + (amount_usd × fx_bps / 10,000)
```

**PAY route:**
```
= $5.00  +  (580 × 50 / 10,000)
= $5.00  +  $2.90
= $7.90  ✓  (matches the record)
```

**ECO route:**
```
= $4.00  +  (580 × 10 / 10,000)
= $4.00  +  $0.58
= $4.58  ✓  (matches the record)
```

The fees come from **pre-agreed bank contracts**. The FX spread is a **fixed rate per corridor**. Pure lookup, zero statistics.

---

## EST_HOURS — rules-based calendar engine

Start with `naive_hours` (raw processing time, no calendar), then add penalties:

| | PAY route | ECO route |
|---|---|---|
| `naive_hours` | 1.5 h | 87.0 h |
| `est_nonbusiness_days` | 0 | 2 (crosses a weekend) |
| `calendar_conflict` | 0 | 0 |
| `hop_count` | 3 | 5 |
| **`est_hours`** | **1.5 h** | **210.48 h** |

**PAY:** 3 hops, no weekend crossed, no conflicts → simulator adds nothing → stays at 1.5 h

**ECO:** 5 hops, crosses 2 non-business days → each non-business day means the payment sits idle waiting for banks to reopen → 87h balloons to 210h

The simulator essentially asks: *"of the raw processing hours, how many fall on bank-closed days?"* and adds that waiting time. Like a parcel delivery app that knows the warehouse is closed on weekends.

---

**Bottom line:** no statistics, no ML. Just:
- **Cost** = fee schedule + FX rate (from contracts)
- **Time** = raw processing hours + calendar waiting time (from bank timetables)